# Kaggle runner v3.1 — E1 pha A (chọn gen), một hoặc nhiều nhóm phương pháp

Gắn một Kaggle Input chứa trực tiếp `experiment-data-perm/`, `experiment-data-geo-target/` và `split_manifests/`
(layout DATA_AUDIT 27/10). Chỉ chạy pha A; pha B (`kaggle_e1_separated_5x5_runner.ipynb`, `RUN_FILTER_EMBEDDED = False`,
`INPUT_SELECTION_ROOT = ...`) đánh giá các panel đã lưu.

v3.1 (Review 29/10, QĐ19): mọi lựa chọn giao thức (k ∈ K_GRID, lọc sơ bộ P0 = 5.000, 13 phương pháp, wrapper chỉ 3 cohort
GPL570 × 3 lần lặp × k ∈ {5, 10, 50}) nằm trong `survey_v2/config.py` đã commit; notebook này chỉ chọn nhóm phương pháp
và cohort. Không truyền `--k`, `--prefilter` ở đây (M18).


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

# Sửa slug này theo Kaggle Input của bạn. Thư mục phải có ba nhánh: experiment-data-perm, experiment-data-geo-target, split_manifests.
INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
COHORTS = ['tcga_luad_lusc_primary_tumor', 'gse10245', 'gse30219', 'gse37745', 'gse42127']

# Bật một hoặc nhiều nhóm. Tất cả dùng cùng cohort, split 5x5 và K_GRID của config (không ghi đè ở đây).
RUN_FILTER = True
RUN_EMBEDDED = True
RUN_CONSTRAINED = True      # abess (cần pip install abess; nếu không cài được script tự bỏ qua và ghi vào config), stability_selection, ktsp
RUN_PANELS = True           # ihc_2, ihc_4 và panel_<tên> từ survey_v2/panels/published_panels.csv (cố định, chạy nhanh)
RUN_WRAPPERS = False        # GA, BPSO: chỉ WRAPPER_DATASETS (3 cohort GPL570), --repeats 0 1 2, k = WRAPPER_K; chạy ở Version riêng
FILTER_METHODS = ['f_test', 'mutual_info', 'mrmr', 'relieff']
EMBEDDED_METHODS = ['l1_lr', 'elastic_net', 'rf_importance', 'svm_rfe']
CONSTRAINED_METHODS = ['abess', 'stability_selection', 'ktsp']
WRAPPER_METHODS = ['ga', 'bpso']
REPEATS = [0, 1, 2, 3, 4]
PREVIOUS_PROGRESS_ARCHIVE = None  # Kaggle Dataset ZIP của output pha A trước đó, nếu resume (checkpoint được bỏ qua khi đã có)
assert RUN_FILTER or RUN_EMBEDDED or RUN_CONSTRAINED or RUN_PANELS or RUN_WRAPPERS, 'Bật ít nhất một nhóm phương pháp'


In [ ]:
# Gate 1 — kiểm tra Input v3, split chính thức và 25 outer folds/cohort.
assert INPUT_V3_DATASET.is_dir(), f'Không thấy Kaggle Input: {INPUT_V3_DATASET}'
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    assert (INPUT_V3_DATASET / folder).is_dir(), f'Thiếu {folder}'
for cohort in COHORTS:
    branch = 'experiment-data-perm' if cohort.startswith('tcga_') else 'experiment-data-geo-target'
    cohort_dir = INPUT_V3_DATASET / branch / cohort
    for filename in ('expression.h5', 'metadata.csv', 'repeated_splits.csv'):
        assert (cohort_dir / filename).is_file(), f'Thiếu {cohort}/{filename}'
print('INPUT_V3_OK')


In [ ]:
# Gate 2 — clone code đã push (provenance dừng nếu HEAD chưa push); symlink Input read-only vào work root để results/ ghi được.
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO_DIR, check=True)
if RUN_CONSTRAINED:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'abess'], check=False)   # không có wheel -> abess bị bỏ qua, có ghi nhận
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    link = WORK_ROOT / folder
    if not link.exists(): link.symlink_to(INPUT_V3_DATASET / folder, target_is_directory=True)
    assert link.resolve() == (INPUT_V3_DATASET / folder).resolve()
if PREVIOUS_PROGRESS_ARCHIVE is not None:
    shutil.unpack_archive(str(PREVIOUS_PROGRESS_ARCHIVE), str(WORK_ROOT))
os.environ['KLTN_DATA_ROOT'] = str(WORK_ROOT)
os.environ['KLTN_STRICT_N'] = '1'   # v3.1: dừng nếu số mẫu mỗi lớp khác C.DATASET_EXPECTED_N (chặn bản dữ liệu sai)
print('COMMIT', commit_hash, 'DATA_ROOT', WORK_ROOT)


In [ ]:
# Pha A. Các nhóm đã bật ghi checkpoint/panel cùng E1_feature_selection để pha B đọc chung.
# Wrapper chạy riêng (E1_wrapper_feature_selection) qua scripts/run_e1_wrappers_all_cohorts.py: chỉ 3 cohort GPL570, 3 lần lặp, k = WRAPPER_K.
methods = []
if RUN_FILTER: methods += FILTER_METHODS
if RUN_EMBEDDED: methods += EMBEDDED_METHODS
if RUN_CONSTRAINED: methods += CONSTRAINED_METHODS
if RUN_PANELS: methods += ['ihc_2', 'ihc_4']
if methods:
    command = [sys.executable, 'scripts/run_e1_feature_selection.py', '--datasets', *COHORTS, '--methods', *methods, '--repeats', *map(str, REPEATS)]
    subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy(), check=True)
if RUN_WRAPPERS:
    command = [sys.executable, 'scripts/run_e1_wrappers_all_cohorts.py', '--methods', *WRAPPER_METHODS]
    subprocess.run(command, cwd=REPO_DIR, env=os.environ.copy(), check=True)


In [ ]:
# Gate 3 — config.json của mỗi cohort phải có traceable = true (tính từ provenance, không gán tay — M18); sau đó archive pha A.
import json
result_root = WORK_ROOT / 'results' / 'survey_v2' / 'E1_feature_selection'
assert any(result_root.glob('*/selected_genes.csv')), 'Không có panel feature selection'
for cfg in result_root.glob('*/config.json'):
    payload = json.loads(cfg.read_text(encoding='utf-8'))
    assert payload.get('traceable') is True, f'{cfg}: traceable = {payload.get("traceable")} -> kết quả KHÔNG được trích'
    assert payload.get('commit_hash') == commit_hash, f'{cfg}: commit_hash khác HEAD đã clone'
archive = Path('/kaggle/working/e1_feature_selection_v3_output.zip')
shutil.make_archive(str(archive.with_suffix('')), 'zip', root_dir=WORK_ROOT, base_dir='results')
print('PHASE_A_OK', result_root, 'ARCHIVE=', archive)
